# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya: FAIR² Dataset Exploration with `mlcroissant`

This notebook demonstrates how to load and explore a Croissant-compliant dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset is described using a Croissant schema and can be accessed via its URL:

`https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json`

In [ ]:
# Ensure `mlcroissant` and other dependencies are installed
!pip install mlcroissant pandas matplotlib seaborn

## 1. Data Loading

Load metadata and records from the dataset using `mlcroissant`. This step will retrieve the dataset schema and make record sets and fields discoverable via their canonical `@id` references.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the Croissant dataset
dataset = mlc.Dataset(croissant_url)

# Access top-level metadata (use object attributes, not dict subscripting)
print(f"Dataset Name: {dataset.metadata.name}\n")
print(f"Description: {dataset.metadata.description}\n")
print(f"Published Date: {dataset.metadata.date_published}")
print("\nAvailable record sets (by @id):", dataset.metadata.record_set)

# Optionally pretty-print more metadata
pp = pprint.PrettyPrinter(indent=2)
print("\nOther metadata fields:")
pp.pprint({
    'Keywords': dataset.metadata.keywords,
    'License': dataset.metadata.license,
    'Spatial coverage': dataset.metadata.spatial_coverage,
    'Temporal coverage': dataset.metadata.temporal_coverage
})

## 2. Data Overview

List all available record sets, their `@id`s, and their fields (also by `@id`). This enables selective data access and reproducible referencing.

In [ ]:
# List all available record sets and their fields (with @id)
if not dataset.metadata.record_set:
    print("No top-level record sets are directly listed in Croissant metadata.\nLet's attempt to enumerate record sets parsed by mlcroissant.")

# Discover available record sets using the live `dataset.record_sets` property:
all_record_sets = list(dataset.record_sets)
if not all_record_sets:
    print("No record sets detected.")
else:
    print("Found record sets:")
    for rs in all_record_sets:
        print(f"  @id: {rs['@id']} | name: {rs.get('name', '(unnamed)')}")
        # List field @ids if available
        if 'field' in rs and rs['field']:
            print("    Fields:")
            fields = rs['field'] if isinstance(rs['field'], list) else [rs['field']]
            for field in fields:
                # The 'field' object may be a dict or a string @id
                if isinstance(field, dict):
                    print(f"      - @id: {field.get('@id', str(field))} (name: {field.get('name', '(unnamed)')})")
                else:
                    print(f"      - @id: {field}")
        else:
            print("    (No field info found)")

## 3. Data Extraction

Load records from a chosen record set (using its `@id`) and convert to a pandas DataFrame. All fields/columns are referenced via their canonical `@id`.

In [ ]:
# Select record sets by @id

# Retrieve all record set @ids present
record_set_ids = [rs['@id'] for rs in dataset.record_sets]
print("Record set @ids:", record_set_ids)

# For demonstration, we'll load the first record set (if present)
dataframes = {}

for record_set_id in record_set_ids:
    print(f"\nLoading records for record set @id: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded {len(df)} records. Columns (field @ids):")
        print(df.columns.tolist())
        print(df.head(2))
    else:
        print("No records found for this record set.")

## 4. Exploratory Data Analysis (EDA)

Demonstrate standard data preprocessing and filtering using field @ids. Steps include filtering records, normalizing numeric fields, and grouping. All column/field references use `@id` for robust, schema-consistent code.

In [ ]:
# Pick the first record set with records for further EDA

selected_rs_id = None
for rsid, df in dataframes.items():
    if not df.empty:
        selected_rs_id = rsid
        break

if selected_rs_id is None:
    raise ValueError("No DataFrame available with records for EDA.")

df = dataframes[selected_rs_id]
print(f"Selected Record Set @id for EDA: {selected_rs_id}")

# List candidate numeric fields (by simple dtype check and/or field name heuristics)
numeric_field_ids = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
print("\nDetected numeric fields (by @id):", numeric_field_ids)

if not numeric_field_ids:
    print("No numeric fields available for EDA.")
else:
    numeric_field_id = numeric_field_ids[0]
    print(f"\nUsing field @id '{numeric_field_id}' for numeric analysis.")
    threshold = df[numeric_field_id].mean()  # Just an example: use mean as threshold
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"Filtered records where {numeric_field_id} > {threshold:.2f}:")
    print(filtered_df.head())

    # Normalize selected numeric field
    norm_col = f"{numeric_field_id}_normalized"
    filtered_df[norm_col] = (
        (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) /
        filtered_df[numeric_field_id].std()
    )
    print(f"\nFirst few normalized values of field '{numeric_field_id}':")
    print(filtered_df[[numeric_field_id, norm_col]].head())

    # Try grouping by a categorical field (heuristic: find a non-numeric field)
    cat_field_candidates = [col for col in df.columns if not pd.api.types.is_numeric_dtype(df[col])]
    group_field = cat_field_candidates[0] if cat_field_candidates else None
    print(f"\nCandidate group field (by @id): {group_field}")
    if group_field and group_field in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean()
        print(f"Mean of '{numeric_field_id}' grouped by '{group_field}':")
        print(grouped_df.head())

## 5. Visualization

Visualize field distributions, relationships, and group-wise aggregates. All axes/data references use field `@id` for clarity.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if selected_rs_id and numeric_field_ids:
    # Histogram of numeric field
    plt.figure(figsize=(7, 4))
    sns.histplot(df[numeric_field_id].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of field '@id': {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

    # If a group field was found, boxplot by group
    if group_field:
        plt.figure(figsize=(10, 5))
        sns.boxplot(x=df[group_field], y=df[numeric_field_id])
        plt.title(f"{numeric_field_id} by {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(numeric_field_id)
        plt.xticks(rotation=30, ha='right')
        plt.show()

## 6. Conclusion

This notebook demonstrated how to load and explore a Croissant-compliant dataset using the `mlcroissant` library for fully reproducible, schema-aware data analysis.

**Key takeaways:**
- All entities, record sets, and fields are referenced by their `@id`, ensuring robust and portable analytical workflows.
- The `mlcroissant` API and Croissant schema enable programmatic discovery of dataset structure and fields.
- Standard EDA and visualizations are easily supported on extracted DataFrames.

Proceed to develop further analyses, modeling, or data integration as needed using this reproducible, machine-actionable foundation.